# Pandas and Exploratory Data Analysis
Companion to the pandas appendix. Sections follow the notes; run in order (later cells use columns created earlier).

## Loading data

In [ ]:
import pandas as pd
import numpy as np

ld = pd.read_csv("../../data/loan_data_train.csv")
print(ld.shape)
ld.head(3)

## First look: info

In [ ]:
ld.info()

## describe and value_counts

In [ ]:
ld.describe().round(1)
ld["Home.Ownership"].value_counts()

## Selecting rows and columns

In [ ]:
ld["Monthly.Income"]                       # one column: a Series
ld[["Monthly.Income", "Home.Ownership"]]   # several columns: a DataFrame
ld.iloc[0:3, 0:4]                          # rows 0-2, columns 0-3, by position
ld.loc[ld["Home.Ownership"] == "RENT", ["ID", "Monthly.Income"]]   # by condition and name
high = ld[(ld["Monthly.Income"] > 10000) & (ld["Home.Ownership"] == "OWN")]
print(len(high))

## Changing columns (notice the '.' in Loan.Length)

In [ ]:
ld = ld.rename(columns={"Monthly.Income": "monthly_income"})
ld["annual_income"] = ld["monthly_income"] * 12
ld = ld.drop(columns=["annual_income"])
ld["Loan.Length"] = ld["Loan.Length"].replace({"60 months": 60, "36 months": 36})
print(ld["Loan.Length"].value_counts(dropna=False))

## Messy text to numbers

In [ ]:
df = pd.DataFrame({
    'Amount': ['$12,500.00', '$4,200.50', 'MISSING', '$8,900.00'],
    'Interest_Rate': ['13.5%', '8.2%', '10.0%', 'N/A']
})
df['Amount'] = df['Amount'].str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')          # 'MISSING' becomes NaN
df['Interest_Rate'] = pd.to_numeric(df['Interest_Rate'].str.replace('%', '', regex=False), errors='coerce')
print(df)
print(df.dtypes)

## String methods

In [ ]:
fico = ld["FICO.Range"].str.split("-", expand=True).astype(float)
ld["fico_mid"] = fico.mean(axis=1)
ld["Employment.Length"].str.extract(r"(\d+)")[0].head(3)
print(ld[["FICO.Range", "fico_mid"]].head(3))

## Dates

In [ ]:
orders = pd.DataFrame({"order_date": ["2024-01-15", "2024-03-02", "2024-12-25"], "amount": [250, 120, 980]})
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["month"] = orders["order_date"].dt.month
orders["weekday"] = orders["order_date"].dt.day_name()
orders["days_since"] = (pd.Timestamp("2025-01-01") - orders["order_date"]).dt.days
print(orders)

## Measuring missing values

In [ ]:
def audit_missingness(df):
    null_counts = df.isnull().sum()
    null_pct = (null_counts / len(df)) * 100
    report = pd.DataFrame({'Missing_Count': null_counts, 'Missing_Pct': null_pct.round(2)})
    return report[report['Missing_Count'] > 0].sort_values(by='Missing_Pct', ascending=False)

print(audit_missingness(ld).head(4))

## Leak-free imputation

In [ ]:
from sklearn.model_selection import train_test_split

ld["Debt.To.Income.Ratio"] = pd.to_numeric(ld["Debt.To.Income.Ratio"].str.replace("%", ""), errors="coerce")
X = ld[["monthly_income", "Debt.To.Income.Ratio", "Home.Ownership"]]
y = ld["Interest.Rate"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)   # 1. split first

train_medians = X_train.median(numeric_only=True)          # 2. statistics from train only
X_train_imputed = X_train.fillna(train_medians)            # 3. same numbers for both
X_test_imputed = X_test.fillna(train_medians)
print(train_medians)

## Assigning with .loc

In [ ]:
df = pd.DataFrame({'Amount': [500, 1500, 2500], 'Interest_Rate': [0.10, 0.12, 0.09]})
df.loc[df['Amount'] > 1000, 'Interest_Rate'] = 0.05
print(df)

## Sorting and counting

In [ ]:
ld.sort_values("monthly_income", ascending=False).head(3)[["ID", "monthly_income"]]
ld["Loan.Purpose"].value_counts(normalize=True).head(3).round(3)

## groupby: customer features

In [ ]:
tx = pd.DataFrame({
    'customer_id': [101, 101, 101, 102, 102, 103],
    'amount': [50.0, 120.0, 30.0, 500.0, 450.0, 15.0],
    'is_late': [0, 1, 0, 0, 0, 0]
})
customer_features = tx.groupby('customer_id').agg(
    total_spend=('amount', 'sum'),
    avg_spend=('amount', 'mean'),
    max_spend=('amount', 'max'),
    num_late_payments=('is_late', 'sum')
).reset_index()
print(customer_features)

## Group-wise imputation, learned on train

In [ ]:
group_medians = X_train.groupby("Home.Ownership")["monthly_income"].median()   # learned on train
overall_median = X_train["monthly_income"].median()

def impute_income(part):
    filled = part["monthly_income"].fillna(part["Home.Ownership"].map(group_medians))
    return filled.fillna(overall_median)                     # categories unseen in train

X_train_imputed["monthly_income"] = impute_income(X_train)
X_test_imputed["monthly_income"] = impute_income(X_test)
print(group_medians.round(0))

## Two-way tables

In [ ]:
pd.crosstab(ld["Home.Ownership"], ld["Loan.Length"], normalize="index").round(2)

## merge

In [ ]:
customers = pd.DataFrame({"customer_id": [101, 102, 104], "city": ["Pune", "Delhi", "Chennai"]})
print(customer_features.merge(customers, on="customer_id", how="left"))
print(customer_features.merge(customers, on="customer_id", how="inner").shape)

## concat

In [ ]:
jan = pd.DataFrame({"customer_id": [101, 102], "amount": [50.0, 500.0]})
feb = pd.DataFrame({"customer_id": [101, 103], "amount": [120.0, 15.0]})
print(pd.concat([jan, feb], ignore_index=True))

## get_dummies and the column mismatch

In [ ]:
train_df = pd.DataFrame({'State': ['CA', 'NY', 'TX', 'WY']})
test_df  = pd.DataFrame({'State': ['CA', 'NY', 'TX']})
train_encoded = pd.get_dummies(train_df, columns=['State'], drop_first=True)
test_encoded  = pd.get_dummies(test_df, columns=['State'], drop_first=True)
print("Train columns:", train_encoded.columns.tolist())
print("Test columns: ", test_encoded.columns.tolist())

## Aligning dummy columns

In [ ]:
test_encoded = test_encoded.reindex(columns=train_encoded.columns, fill_value=0)
assert list(train_encoded.columns) == list(test_encoded.columns)
print(test_encoded.columns.tolist())

## EDA: the target

In [ ]:
bd = pd.read_csv("../../data/bd_train.csv")
bd["high_revenue"] = (bd["Revenue.Grid"] == 1).astype(int)
print(bd.shape)
print(bd["high_revenue"].value_counts(normalize=True).round(3))

## EDA: one column at a time

In [ ]:
print(bd[["post_code", "post_area", "TVarea", "family_income"]].nunique())
bd = bd.drop(columns=["post_code", "post_area"])     # too many unique values to be useful
print((bd["year_last_moved"] == 0).sum())
num = ["Average.Credit.Card.Transaction", "Term.Deposit", "Life.Insurance",
       "Online.Purchase.Amount", "Portfolio.Balance"]
print((bd[num] == 0).mean().round(2))
print((bd["Portfolio.Balance"] < 0).sum())

## EDA: columns against the target

In [ ]:
print(bd.groupby("high_revenue")[num].median().round(1).T)
rate = bd.groupby("region")["high_revenue"].agg(["mean", "count"]).round(3)
print(rate.sort_values("count").head(3))

## EDA: plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.histplot(bd["Online.Purchase.Amount"], bins=60)
plt.show()
sns.barplot(x="age_band", y="high_revenue", data=bd, errorbar=None)
plt.xticks(rotation=60); plt.show()
sns.boxplot(x="high_revenue", y=np.log1p(bd["Online.Purchase.Amount"]), data=bd)
plt.show()
sns.heatmap(bd[num].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.show()

## Memory: downcasting

In [ ]:
def downcast_dtypes(df):
    df = df.copy()
    for col in df.columns:
        col_type = df[col].dtype
        if col_type == 'float64':
            df[col] = pd.to_numeric(df[col], downcast='float')
        elif col_type == 'int64':
            df[col] = pd.to_numeric(df[col], downcast='integer')
        elif col_type == 'object':
            if df[col].nunique() / len(df[col]) < 0.5:      # low cardinality: store as category
                df[col] = df[col].astype('category')
    return df

before = bd.memory_usage(deep=True).sum() / 1e6
after = downcast_dtypes(bd).memory_usage(deep=True).sum() / 1e6
print(f"{before:.1f} MB -> {after:.1f} MB")

## Try it yourself
1. How many `Amount.Requested` values fail to convert to numbers, and what are they?
2. Highest high-revenue rate among occupations with at least 100 customers?
3. Replace zeros in `year_last_moved` with NaN; create `years_at_address` relative to 2000.
4. `pd.crosstab(bd['children'], bd['high_revenue'], normalize='index')`: any pattern?
5. Merge `customer_features` with `customers` using `how='outer'`. How many rows and why?